In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_percentage_error as MAPE
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
import statsmodels.api as sm
from statsmodels.api import OLS
from statsmodels.api import add_constant
import pickle

import os
import logging

In [15]:
# In Python, calling logging.getLogger(name) multiple times with the exact same name will neither overwrite the logger nor create multiple copies
# ; instead, it will always return a reference to the exact same singleton object. 

# However, how you configure that logger can cause an unexpected side effect: it will create multiple copies of your log outputs (duplicated logs)
# if you keep adding handlers to it

logger_name = '6_Model_Prediction'

def create_logger_handler(log_folder, logger_name):
    # Ensure log directory exists
    log_dir = log_folder
    os.makedirs(log_dir, exist_ok=True)

    # logging configuration
    logger = logging.getLogger(logger_name)
    logger.setLevel('DEBUG')


    # This is to print at console level
    console_handler = logging.StreamHandler()
    console_handler.setLevel('DEBUG')

  # This is to print in log file
    file_path = os.path.join(log_dir, logger_name+'.log')
    file_handler = logging.FileHandler(file_path)
    file_handler.setLevel('DEBUG')

    formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
    console_handler.setFormatter(formatter)
    file_handler.setFormatter(formatter)

    logger.addHandler(console_handler)
    logger.addHandler(file_handler)

    return logger

logger = create_logger_handler('../logs', logger_name)

# If we want to diable the logger
# logging.getLogger("Data_Ingestion").disabled = True

# If we disable logger then we can again enable it by just triggering above cell again. Logger will automatically enable it.

In [14]:
# If we have run above code multiple times then we will have multiple handlers which results in printing same output twice/thrice
# We can remove active handlers but we can't remove logger

def close_remove_active_handler(logger_name):

    # active log handlers
    logger = logging.getLogger(logger_name)
    print(f'Active handlers for logger- {logger_name} is -', len(logger.handlers))

    for handler in logger.handlers[:]:
        handler.close()
        logger.removeHandler(handler)

    print(f'Active handlers for logger- {logger_name} is -', len(logger.handlers))

close_remove_active_handler(logger_name)


Active handlers for logger- 6_Model_Prediction is - 2
Active handlers for logger- 6_Model_Prediction is - 0


In [ ]:
def load_model(model_path):
    """ Loading Model """

    try:
        with open(model_path, 'rb') as file:
            model = pickle.load(file)
        logger.debug('Model has loaded successfully') 
        return model   

    except Exception as e:
        logger.debug('Model loading was unsuccessfull - ', e)
        raise


def load_inference_data(path):
    """ Loading data-set on which we have to make prediction """

    try:
        df = pd.read_csv(path)
        logger.debug('Inference data has loaded successfully')
        
        return df

    except Exception as e:
        logger.debug("Loading test data has failed - %s", e)
        raise


def pre_processing(inference_data):

    """ Data pre-processing on test data"""
    
    try:
        inference_data_constant = add_constant(inference_data)
        logger.debug('Constant has been added to inference_data')

        return inference_data_constant
    
    except Exception as e:
        logger.debug('x_test pre-processing has failed - %s', e)
        raise


def prediction(model, data):
    """ This function is predict on inference data"""

    try:
        prediction = model.predict(data).tolist()
        logger.debug('Prediction has been generated successfully')

        return prediction
    
    except Exception as e:
        logger.debug('Unexpected error in generating prediction - %s', e)




In [ ]:
model = load_model('../models/Regression.pkl')

inference_data = load_inference_data('../data/Inference/Inference_Data.csv')

processed_data = pre_processing(inference_data)

prediction = prediction(model, processed_data)

In [32]:
output_df = inference_data.copy()
output_df['Prediction'] = np.round(prediction,2)
output_df.head()

,Education,Experience,Age,Prediction
0,16,6,28,10496.21
1,12,3,21,6382.44
2,12,8,26,6957.59
3,5,44,55,4503.45
4,14,16,36,9762.16
